nnUNetv2_predict parameters :
--disable_tta
-step_size <value> (Default is 0.5, 1.0 is 0% overlap)
-npp (preprocessing processes) and -nps (segmentation export processes). Both default to 3.

predict on CPU : 
-device cpu
recommanded to set --disable_tta -step_size 0.75

export nnUNet_raw="/nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/nnunet/nnUNet_raw"
export nnUNet_preprocessed="/nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/nnunet/nnUNet_preprocessed"
export nnUNet_results="/nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/nnunet/nnUNet_results"

# run inference on 10 subject with time : 

time nnUNetv2_predict \
  -i /nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/nnunet/nnUNet_raw/Dataset003_oppscreen_20_subjects/imagesTs_10_subjects \
  -o /nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/nnunet/nnUNet_raw/Dataset003_oppscreen_20_subjects/predsTs_10_subjects \
  -d 3 \
  -c 3d_fullres \
  -f all \
  -tr nnUNetTrainer \
  -p nnUNetResEncUNetPlans_40G \
  -chk checkpoint_best.pth \
  -step_size 0.75

1) nb processes : 
ran on thor, 20C, 64GB RAM
check with : 
watch -n 1 "ps -u dpxuser -o rss,cmd | grep -E 'nnUNet|spawn_main' | grep -v grep | awk '{sum+=\$1} END {printf \"Total nnUNet RAM: %.2f GB\n\", sum/1024/1024}'"

metrics : 
default params : ~10 GB RAM, ~10 GB VRAM, 5m28s, val dice 0.8644
-npp 6 -nps 6 : ~15 GB RAM, ~10 GB VRAM, 5m26

2) --disable_tta : 
~12 GB RAM, ~8 GB VRAM, 2m39, val dice 0.8615
dont forget to eval results : 
python scripts/nnunet_patchwork_comp/eval.py \
    --gt-pattern   /nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/nnunet/nnUNet_raw/Dataset003_oppscreen_20_subjects/labelsTs_10_subjects/{subject}.nii.gz \
    --pred-pattern /nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/nnunet/nnUNet_raw/Dataset003_oppscreen_20_subjects/predsTs_10_subjects/{subject}.nii.gz \
    --subjects     /nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/nnunet/nnUNet_raw/Dataset003_oppscreen_20_subjects/labelsTs_10_subjects \
    --output       results/inference_benchmark/nnunet_step_size_0_75.json --no-nsd

3) -step_size
0.75 : ~10 GB RAM, ~10 GB VRAM, 5m21, val dice 0.8644
1 : ~10 GB RAM, ~10 GB VRAM, 4m27, val dice 0.8603

--disable_tta -step_size 1
~15 GB RAM, ~10 GB VRAM, 2m35, val dice 0.8561

3) -device cpu --disable_tta -step_size 1 
~13 GB RAM, 50m24, val dice 0.8644